# Лабораторная работа №10. Транспортный уровень и UDP

Ports, sockets, multiplexing и реальная UDP datagram. Главный вопрос: какие гарантии предоставляет UDP — и каких гарантий там нет.

## Схема

![UDP](assets/udp.svg)

Два Linux-хоста на одном `/24`. На `pc2` поднимем UDP listener, на `pc1` отправим несколько сообщений с одного destination port.

## 1. Настроим адреса

`pc1`:

```bash
ip addr flush dev eth1
ip addr add 10.10.0.10/24 dev eth1
```

`pc2`:

```bash
ip addr flush dev eth1
ip addr add 10.10.0.20/24 dev eth1
```

Проверьте `ping -c 1 10.10.0.20`, чтобы отделить L3-проблемы от transport experiment.

### Q1. Зачем transport layer нужны ports, если IP уже однозначно определяет destination host?

`введите ваш ответ`

## 2. Откроем UDP socket

В terminal `pc2`:

```bash
nc -u -l -p 9000
```

В другом terminal `pc1`:

```bash
printf 'one\n' | nc -u -w 1 10.10.0.20 9000
printf 'two\n' | nc -u -w 1 10.10.0.20 9000
```

Если ваш `nc` после первого сообщения завершил listener, запустите listener ещё раз. Это поведение конкретной утилиты, а не UDP protocol.

### Q2. Где в UDP находится признак «это первое сообщение соединения»? Если такого поля нет, что это означает для приложения?

`введите ваш ответ`

In [ ]:
%%capture_traffic pc1:eth1 --filter "udp.port == 9000" --timeout 12 --packets 100 --save captures/udp.pcap
import time
print("Отправьте 2–3 UDP сообщения с pc1 на pc2:9000")
time.sleep(10)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/udp.pcap", "udp.port == 9000", limit=30)
frames = viewer.packets()
display(frames)
if not frames.empty:
    print(viewer.packet(int(frames.iloc[0]["№"]), "protocols"))

## 3. Четыре поля header

В protocol tree найдите:

```text
Source Port
Destination Port
Length
Checksum
```

Нет sequence number, acknowledgement number, window и handshake flags. Если приложению нужны ordering/retry/reliability, оно должно реализовать это выше UDP или выбрать другой transport.

### Q3. Может ли UDP datagram прийти дважды, потеряться или прийти после более поздней datagram? Как это должно влиять на дизайн приложения?

`введите ваш ответ`

## 4. Multiplexing

На `pc2` посмотрите sockets:

```bash
ss -lunp
```

IP приводит packet на host, destination port — к конкретному socket/application endpoint.

## CHECK

Checker проверит IP-состояние и сам выполнит небольшой UDP exchange между `pc1` и `pc2`; ручной listener из notebook не обязан оставаться запущенным.

## Итог

UDP даёт datagram abstraction, ports и checksum, но не создаёт connection state, ordering или retransmission. Простота header означает больше ответственности у приложения.